# CO2 Awareness Experiment - Data Analysis

This notebook analyzes data from participants in the CO2 awareness experiment.

**Key Research Questions:**
1. Did participants improve their CO2 knowledge after the learning intervention?
2. Does the effect differ by demographics (age, gender, diet)?
3. What do participants report learning from the experience?

**Data Structure:**
- 6 quiz scores (pre/post for generic, AH-specific, and personal products)
- Demographics (age, gender, diet, shopping frequency)
- Likert questionnaires (pre & post attitudes)
- Open-ended reflection responses

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import requests
import json
from datetime import datetime

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

print("Libraries loaded successfully!")

## 2. Load Data from API

Fetch experiment data from the local server. Make sure the server is running (`npm run dev` in sustainable-shop-webapp).

In [ ]:
# Configuration
SERVER_URL = "http://localhost:3001"
COMPLETED_ONLY = True  # Set to False to include incomplete sessions

# Fetch data from API
def fetch_data(completed_only=True):
    url = f"{SERVER_URL}/api/admin/experiment/export"
    params = {'completedOnly': 'true'} if completed_only else {}
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.ConnectionError:
        print(f"❌ Could not connect to server at {SERVER_URL}")
        print("   Make sure the server is running: cd sustainable-shop-webapp && npm run dev")
        return None

# Load data
raw_data = fetch_data(COMPLETED_ONLY)

if raw_data:
    print(f"✅ Loaded {raw_data['count']} sessions ({raw_data.get('completedCount', 'N/A')} completed)")
    print(f"   Exported at: {raw_data['exportedAt']}")

In [ ]:
# Convert to DataFrame
sessions = raw_data.get('sessions', [])

# Extract flat fields (exclude raw JSONB columns starting with _)
flat_data = []
for s in sessions:
    row = {k: v for k, v in s.items() if not k.startswith('_')}
    flat_data.append(row)

df = pd.DataFrame(flat_data)

# Convert timestamps
for col in ['started_at', 'completed_at', 'updated_at']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"DataFrame shape: {df.shape}")
df.head()

## 3. Data Inspection

Understand the structure and quality of the data.

In [ ]:
# Basic info about the dataset
print("=== Dataset Info ===")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn types:")
df.info()

In [ ]:
# Check key columns for missing values
key_columns = ['quiz1_score', 'quiz2_score', 'quiz3_score', 'quiz4_score', 
               'quiz5_score', 'quiz6_score', 'demo_age', 'demo_gender', 'demo_diet']

print("=== Missing Values in Key Columns ===")
missing = df[key_columns].isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values in key columns!")

print("\n=== Quiz Score Ranges ===")
quiz_cols = [c for c in df.columns if 'quiz' in c and 'score' in c]
df[quiz_cols].describe().round(1)

## 4. Demographics Overview

Understand who participated in the experiment.

In [ ]:
# Demographics breakdown
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gender distribution
if 'demo_gender' in df.columns:
    gender_counts = df['demo_gender'].value_counts()
    axes[0, 0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)
    axes[0, 0].set_title('Gender Distribution')

# Diet distribution
if 'demo_diet' in df.columns:
    diet_counts = df['demo_diet'].value_counts()
    sns.barplot(x=diet_counts.index, y=diet_counts.values, ax=axes[0, 1], palette='viridis')
    axes[0, 1].set_title('Diet Distribution')
    axes[0, 1].set_xlabel('Diet Type')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].tick_params(axis='x', rotation=45)

# Age distribution
if 'demo_age' in df.columns:
    df['demo_age_numeric'] = pd.to_numeric(df['demo_age'], errors='coerce')
    ages = df['demo_age_numeric'].dropna()
    axes[1, 0].hist(ages, bins=15, edgecolor='black', alpha=0.7)
    axes[1, 0].set_title(f'Age Distribution (mean={ages.mean():.1f}, n={len(ages)})')
    axes[1, 0].set_xlabel('Age')
    axes[1, 0].set_ylabel('Count')

# Shopping frequency
if 'demo_shopping_frequency' in df.columns:
    freq_counts = df['demo_shopping_frequency'].value_counts()
    sns.barplot(x=freq_counts.index, y=freq_counts.values, ax=axes[1, 1], palette='coolwarm')
    axes[1, 1].set_title('Shopping Frequency')
    axes[1, 1].set_xlabel('Frequency')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Summary table
print("\n=== Demographics Summary ===")
for col in ['demo_gender', 'demo_diet', 'demo_shopping_frequency']:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts())

## 5. Learning Effect Analysis (Pre vs Post)

**Key hypothesis:** Participants score higher on post-intervention quizzes than pre-intervention quizzes.

Quiz pairs:
- **Generic products:** Quiz 1 (pre) → Quiz 3 (post)
- **AH-specific products:** Quiz 5 (pre) → Quiz 6 (post)  
- **Personal products:** Quiz 2 (pre) → Quiz 4 (post)

We use **paired t-tests** since each participant has both pre and post scores.

In [ ]:
def analyze_learning_effect(df, pre_col, post_col, label):
    """Perform paired t-test and calculate effect size for pre/post comparison."""
    # Get paired data (both pre and post available)
    paired = df[[pre_col, post_col]].dropna()
    
    if len(paired) < 3:
        return {'label': label, 'error': f'Insufficient data (n={len(paired)})'}
    
    pre = paired[pre_col]
    post = paired[post_col]
    
    # Paired t-test
    t_stat, p_value = stats.ttest_rel(post, pre)
    
    # Wilcoxon signed-rank test (non-parametric)
    try:
        w_stat, w_pvalue = stats.wilcoxon(post - pre)
    except:
        w_stat, w_pvalue = None, None
    
    # Effect size (Cohen's d for paired samples)
    diff = post - pre
    cohens_d = diff.mean() / diff.std() if diff.std() > 0 else 0
    
    return {
        'label': label,
        'n': len(paired),
        'pre_mean': pre.mean(),
        'pre_std': pre.std(),
        'post_mean': post.mean(),
        'post_std': post.std(),
        'improvement': post.mean() - pre.mean(),
        't_statistic': t_stat,
        'p_value': p_value,
        'wilcoxon_p': w_pvalue,
        'cohens_d': cohens_d,
        'significant': p_value < 0.05
    }

# Analyze all three quiz pairs
quiz_pairs = [
    ('quiz1_score', 'quiz3_score', 'Generic Products'),
    ('quiz5_score', 'quiz6_score', 'AH-Specific Products'),
    ('quiz2_score', 'quiz4_score', 'Personal Products'),
]

results = []
for pre, post, label in quiz_pairs:
    result = analyze_learning_effect(df, pre, post, label)
    results.append(result)

# Display results
results_df = pd.DataFrame(results)
print("=" * 80)
print("LEARNING EFFECT ANALYSIS - PAIRED T-TEST RESULTS")
print("=" * 80)

for r in results:
    if 'error' in r:
        print(f"\n{r['label']}: {r['error']}")
        continue
    
    sig_marker = "***" if r['p_value'] < 0.001 else "**" if r['p_value'] < 0.01 else "*" if r['p_value'] < 0.05 else "ns"
    effect_size = "large" if abs(r['cohens_d']) >= 0.8 else "medium" if abs(r['cohens_d']) >= 0.5 else "small" if abs(r['cohens_d']) >= 0.2 else "negligible"
    
    print(f"\n{'─' * 60}")
    print(f"📊 {r['label']} (n={r['n']})")
    print(f"{'─' * 60}")
    print(f"   Pre-test:     {r['pre_mean']:.1f} ± {r['pre_std']:.1f}")
    print(f"   Post-test:    {r['post_mean']:.1f} ± {r['post_std']:.1f}")
    print(f"   Improvement:  {r['improvement']:+.1f} points")
    print(f"")
    print(f"   t-statistic:  {r['t_statistic']:.3f}")
    print(f"   p-value:      {r['p_value']:.4f} {sig_marker}")
    print(f"   Cohen's d:    {r['cohens_d']:.3f} ({effect_size} effect)")

print(f"\n{'=' * 80}")
print("Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")

## 6. Data Visualization

Visual comparison of pre vs post quiz scores.

In [ ]:
# Box plots comparing pre vs post scores
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

quiz_pairs = [
    ('quiz1_score', 'quiz3_score', 'Generic Products'),
    ('quiz5_score', 'quiz6_score', 'AH-Specific Products'),
    ('quiz2_score', 'quiz4_score', 'Personal Products'),
]

colors = ['#ff6b6b', '#4ecdc4']  # red for pre, teal for post

for ax, (pre, post, title) in zip(axes, quiz_pairs):
    data_to_plot = []
    labels = []
    
    pre_data = df[pre].dropna()
    post_data = df[post].dropna()
    
    bp = ax.boxplot([pre_data, post_data], labels=['Pre', 'Post'], patch_artist=True)
    
    # Color the boxes
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Add mean markers
    ax.scatter([1, 2], [pre_data.mean(), post_data.mean()], 
               color='black', marker='D', s=50, zorder=3, label='Mean')
    
    # Calculate improvement
    improvement = post_data.mean() - pre_data.mean()
    ax.set_title(f'{title}\n(+{improvement:.1f} points)', fontsize=12)
    ax.set_ylabel('Score (0-100)')
    ax.set_ylim(0, 105)
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.3, label='Random (50)')

plt.suptitle('Pre vs Post Intervention Quiz Scores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Individual improvement distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

improvement_cols = [
    ('generic_improvement', 'Generic Products'),
    ('ah_improvement', 'AH-Specific Products'),
    ('personal_improvement', 'Personal Products'),
]

for ax, (col, title) in zip(axes, improvement_cols):
    if col in df.columns:
        data = df[col].dropna()
        
        # Histogram
        ax.hist(data, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
        ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No change')
        ax.axvline(x=data.mean(), color='green', linestyle='-', linewidth=2, 
                   label=f'Mean: {data.mean():.1f}')
        
        ax.set_title(f'{title}\n(n={len(data)})')
        ax.set_xlabel('Score Improvement (Post - Pre)')
        ax.set_ylabel('Count')
        ax.legend()

plt.suptitle('Distribution of Individual Learning Effects', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# What percentage improved?
print("\n=== Percentage of Participants Who Improved ===")
for col, label in improvement_cols:
    if col in df.columns:
        data = df[col].dropna()
        pct_improved = (data > 0).mean() * 100
        pct_same = (data == 0).mean() * 100  
        pct_worse = (data < 0).mean() * 100
        print(f"{label}: {pct_improved:.1f}% improved, {pct_same:.1f}% same, {pct_worse:.1f}% worse")

## 7. Learning Effect by Demographics

Does the intervention work better for certain groups?

In [ ]:
# Learning effect by diet type
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

improvement_cols = ['generic_improvement', 'ah_improvement', 'personal_improvement']
titles = ['Generic Products', 'AH-Specific Products', 'Personal Products']

for ax, col, title in zip(axes, improvement_cols, titles):
    if col in df.columns and 'demo_diet' in df.columns:
        # Group by diet
        diet_data = df.groupby('demo_diet')[col].agg(['mean', 'std', 'count']).reset_index()
        diet_data = diet_data[diet_data['count'] >= 2]  # At least 2 participants
        
        if len(diet_data) > 0:
            bars = ax.bar(diet_data['demo_diet'], diet_data['mean'], 
                         yerr=diet_data['std'], capsize=5, 
                         color='steelblue', alpha=0.7, edgecolor='black')
            
            # Add count labels
            for i, (idx, row) in enumerate(diet_data.iterrows()):
                ax.text(i, row['mean'] + row['std'] + 2, f"n={int(row['count'])}", 
                       ha='center', fontsize=9)
            
            ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
            ax.set_title(title)
            ax.set_xlabel('Diet Type')
            ax.set_ylabel('Mean Improvement')
            ax.tick_params(axis='x', rotation=45)

plt.suptitle('Learning Effect by Diet Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Questionnaire Analysis

Analyze attitudes and self-perceptions before and after the intervention.

**Pre-questionnaire (1-5 Likert):**
- pre_q1: I consider my food choices environmentally sustainable
- pre_q2: I feel confident in my knowledge of food environmental impact
- pre_q3: I trust eco-labels when buying food
- pre_q4: Eco-labels are easy to understand
- pre_q5: I find it easy to compare products by environmental impact
- pre_q6: I actively consider environmental impact when buying food

**Post-questionnaire (1-5 Likert):**
- post_q1: Better understanding of environmental impact after study
- post_q2: The ranking system was clear and easy to understand
- post_q3: I trust the CO₂ ranking system presented
- post_q4: This system is clearer than existing eco-labels
- post_q5: Feedback on personal purchases was useful
- post_q6: Would use this information for future choices
- post_q7: Quizzes helped me learn about environmental impact

In [ ]:
# Questionnaire response analysis
pre_q_cols = [c for c in df.columns if c.startswith('pre_q') and c[5:].isdigit()]
post_q_cols = [c for c in df.columns if c.startswith('post_q') and c[6:].isdigit()]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pre-questionnaire
if pre_q_cols:
    pre_means = df[pre_q_cols].mean()
    pre_stds = df[pre_q_cols].std()
    
    x = range(len(pre_q_cols))
    axes[0].bar(x, pre_means, yerr=pre_stds, capsize=3, color='coral', alpha=0.7, edgecolor='black')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([c.replace('pre_', '') for c in pre_q_cols])
    axes[0].set_ylabel('Mean Score (1-5 Likert)')
    axes[0].set_title('Pre-Questionnaire Responses')
    axes[0].set_ylim(1, 5.5)
    axes[0].axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Neutral (3)')

# Post-questionnaire
if post_q_cols:
    post_means = df[post_q_cols].mean()
    post_stds = df[post_q_cols].std()
    
    x = range(len(post_q_cols))
    axes[1].bar(x, post_means, yerr=post_stds, capsize=3, color='teal', alpha=0.7, edgecolor='black')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([c.replace('post_', '') for c in post_q_cols])
    axes[1].set_ylabel('Mean Score (1-5 Likert)')
    axes[1].set_title('Post-Questionnaire Responses')
    axes[1].set_ylim(1, 5.5)
    axes[1].axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Neutral (3)')

plt.tight_layout()
plt.show()

# Summary statistics
print("=== Questionnaire Summary Statistics ===")
print("\nPre-Questionnaire:")
print(df[pre_q_cols].describe().round(2).T[['mean', 'std', 'min', 'max']])
print("\nPost-Questionnaire:")
print(df[post_q_cols].describe().round(2).T[['mean', 'std', 'min', 'max']])

## 9. Qualitative Responses

Open-ended reflection responses for thematic analysis.

In [ ]:
# Extract qualitative responses from raw data
qualitative_responses = []

for session in sessions:
    reflection = session.get('_post_questionnaire_open') or session.get('_reflection') or {}
    if reflection and isinstance(reflection, dict):
        entry = {
            'session_id': session.get('id', '')[:8],  # First 8 chars for anonymity
            'diet': session.get('demo_diet'),
            'generic_improvement': session.get('generic_improvement'),
        }
        # Extract text responses
        for key, value in reflection.items():
            if isinstance(value, str) and value.strip():
                entry[key] = value.strip()
        
        if len(entry) > 3:  # Has actual text responses
            qualitative_responses.append(entry)

print(f"Found {len(qualitative_responses)} responses with open-ended feedback\n")

# Display responses by question type
question_keys = ['ref_reflection', 'ref_surprise', 'ref_system_feedback', 
                 'ref_trust_comparison', 'ref_improvement']

for key in question_keys:
    responses_for_key = [(r['session_id'], r.get(key)) for r in qualitative_responses if r.get(key)]
    if responses_for_key:
        print(f"{'=' * 70}")
        print(f"📝 {key.replace('ref_', '').replace('_', ' ').upper()}")
        print(f"   ({len(responses_for_key)} responses)")
        print(f"{'=' * 70}")
        for session_id, response in responses_for_key[:5]:  # Show first 5
            print(f"\n[{session_id}]: {response[:300]}{'...' if len(response) > 300 else ''}")
        if len(responses_for_key) > 5:
            print(f"\n... and {len(responses_for_key) - 5} more responses")
        print()

## 10. Export Data

Save processed data for further analysis (SPSS, Excel, etc.)

In [ ]:
# Export options
OUTPUT_DIR = "analysis_output"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Save flat DataFrame to CSV (for SPSS/Excel)
csv_path = f"{OUTPUT_DIR}/experiment_data.csv"
df.to_csv(csv_path, index=False)
print(f"✅ Saved: {csv_path}")

# 2. Save qualitative responses to JSON
qual_path = f"{OUTPUT_DIR}/qualitative_responses.json"
with open(qual_path, 'w', encoding='utf-8') as f:
    json.dump(qualitative_responses, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: {qual_path}")

# 3. Save statistical results
stats_df = pd.DataFrame(results)
stats_path = f"{OUTPUT_DIR}/statistical_results.csv"
stats_df.to_csv(stats_path, index=False)
print(f"✅ Saved: {stats_path}")

# 4. Save full raw data (with nested quiz details)
full_path = f"{OUTPUT_DIR}/experiment_data_full.json"
with open(full_path, 'w', encoding='utf-8') as f:
    json.dump(raw_data, f, indent=2, default=str)
print(f"✅ Saved: {full_path}")

print(f"\n📁 All files saved to: {os.path.abspath(OUTPUT_DIR)}")

## Summary

Key findings from this analysis:

1. **Learning Effect**: Check the paired t-test results above to see if scores improved significantly
2. **Effect Size**: Cohen's d indicates the practical significance of the improvement
3. **Demographics**: Review if certain groups benefit more from the intervention
4. **User Feedback**: Qualitative responses provide insight into user experience

---
*Generated by experiment_analysis.ipynb*